In [0]:
%sql
SELECT * FROM careconnect.default.users;

In [0]:
%sql
USE CATALOG careconnect;
USE SCHEMA default;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS users (
    user_id STRING,
    first_name STRING,
    last_name STRING,
    age INT,
    gender STRING,
    email STRING,
    if_pregnant BOOLEAN
)
USING DELTA;


In [0]:
%sql
CREATE TABLE IF NOT EXISTS medical_conditions (
    condition_id STRING,
    user_id STRING,
    condition_name STRING,
    diagnosis_date DATE,
    is_chronic BOOLEAN,
    notes STRING
)
USING DELTA;


In [0]:
%sql
SELECT * FROM careconnect.default.medical_conditions;


In [0]:
%sql 
SELECT 
  STATE,
  COUNT(*) AS hospital_count
FROM careconnect.default.all_india_hospital_list
GROUP BY STATE
ORDER BY hospital_count DESC;

# SQl agent 

In [0]:
system_message = """You are an intelligent SQL assistant for a healthcare data analytics platform running inside Databricks notebooks.

Your job is to translate user questions into **pure SQL queries** that are ready to run in `%sql` cells inside Databricks.

### Available Tables
The following tables are available in Unity Catalog under catalog `careconnect` and schema `default`.

1. Table: `careconnect.default.all_india_hospital_list`
   - Columns:
     - Hospital Name (string)
     - Address (string)
     - CITY (string)
     - STATE (string)
     - PIN CODE (string)
     - PPN / NON PPN (string)
   - Description: Contains hospital information across India, useful for analyzing location, type (PPN/NON PPN), and accessibility of hospitals.

2. Table: `careconnect.default.medical_conditions`
   - Columns:
     - condition_id (string)
     - user_id (string)
     - condition_name (string)
     - diagnosis_date (date)
     - is_chronic (boolean)
     - notes (string)
   - Description: Contains medical condition records for users, including diagnosis date and chronic status. 

3. Table: `careconnect.default.users`
   - Columns:
     - user_id (string)
     - first_name (string)
     - last_name (string)
     - age (bigint)
     - gender (string)
     - email (string)
     - if_pregnant (boolean)
   - Description: Contains information about individual users including demographics and health status. It supports tracking patient details and tailoring healthcare services.

### Important Relationships
- The `user_id` column in the `users` table uniquely identifies each user.
- The `medical_conditions` table can have multiple rows per user because a user may have multiple medical conditions. Thus, `user_id` in `medical_conditions` is not unique.
- Each row in `medical_conditions` is uniquely identified by the `condition_id` column.
- Use these relationships to join tables when answering questions involving both users and their medical conditions.

### Rules:
- Only generate **pure SQL queries** that will be executed in a `%sql` cell in a Databricks notebook.
- Use **fully qualified table names** as listed above.
- Do not explain the query.
- Always wrap the SQL in triple backticks (```).
- Use `LIMIT` in queries that return large result sets.
- Make intelligent assumptions if some information is missing or unclear.

Respond only with the SQL code block, and nothing else.

"""

In [0]:
from openai import OpenAI

DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url="https://dbc-65fcd381-2e74.cloud.databricks.com/serving-endpoints"
)

def generate_sql_from_question(question):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": question}
    ]
    
    response = client.chat.completions.create(
        model="databricks-claude-3-7-sonnet",
        messages=messages
    )
    
    content = response.choices[0].message.content
    sql_code = content.split("```")[1].strip() if "```" in content else content
    return sql_code

def ask_and_run(question):
    sql_query = generate_sql_from_question(question)
    sql_query= sql_query.replace("sql",'') + ";"
    try:
        df = spark.sql(sql_query)
        return df
    except Exception as e:
        print("Failed to run SQL:\n", e)

# Expensive task run with patient 

In [0]:
# question = "Show the number of hospitals per state sorted by highest first"
# ask_and_run(question)